In [1]:
import json
import pandas as pd

In [2]:
data = []
with open('../data/games_informations/example.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        appid, info = next(iter(item.items()))
        info['appid'] = appid
        data.append(info)


In [3]:
df = pd.DataFrame(data)
cols_to_drop = ['header_image', 'reviews', 'support_info', 'game_link', 'appid', 'demos', 'ext_user_account_notice', 'drm_notice']
for col in cols_to_drop:
    if col in df.columns:
        df = df.drop(col, axis=1)
if df['achievements'].any():
    df['achievements'] = 1
else:
    df['achievements'] = 0
if df['dlc'].any():
    df['dlc'] = 1
else:
    df['dlc'] = 0

print(df.head)

<bound method NDFrame.head of                                            name  is_free  \
0                                  Mystic Space    False   
1                                     DK Online     True   
2                                 MissionX Beta     True   
3                                          Evie    False   
4                 Winner Winner Chicken Dinner!    False   
5                                      Later On    False   
6                                      Creatura    False   
7                                    Technicity    False   
8                            Beat the Rhythm VR    False   
9                                      Drakkhen    False   
10                                     Mystical    False   
11                                      Eternam    False   
12                                   Marco Polo    False   
13                                Chaos Control    False   
14                    Time Gate: Knight's Chase    False   
15        

In [ ]:
pattern = r'<[^>]*>'
df['about_the_game'] = df['about_the_game'].str.replace(pattern, ' ', regex=True).replace(r'\n', ' ', regex=True)
df['supported_languages'] = df['supported_languages'].str.replace(pattern, ' ', regex=True).str.replace('*', ' ')

df['price_overview'] = df['price_overview'].apply(
    lambda x: f"{x['currency']} {x['initial']} {x['final']}" if isinstance(x, dict) else None
)

df['developers'] = df['developers'].apply(lambda x: " ".join(x) if isinstance(x, list) else x)
df['publishers'] = df['publishers'].apply(lambda x: " ".join(x) if isinstance(x, list) else x)

# Extract description values from genres and categories
df['genres'] = df['genres'].apply(
    lambda x: ", ".join([d.get('description', '') for d in x]) if isinstance(x, list) else x
)
df['categories'] = df['categories'].apply(
    lambda x: ", ".join([d.get('description', '') for d in x]) if isinstance(x, list) else x
)

def extract_release_date(x):
    # If the cell is re-run, it might already be parsed into a string. 
    # We just return it directly instead of turning it into None.
    if isinstance(x, str):
        return x
    if not isinstance(x, dict):
        return None
    coming_soon = 1 if x.get('coming_soon') else 0
    date_str = x.get('date', '')
    try:
        # Tries to parse date and output as DD/MM/YYYY
        date_formatted = pd.to_datetime(date_str).strftime('%d/%m/%Y')
    except Exception:
        # Fallback to original string if pandas cannot parse the date
        date_formatted = date_str
    return f"{coming_soon} {date_formatted}"

df['release_date'] = df['release_date'].apply(extract_release_date)

df['supported_languages'] = df['supported_languages'].str.replace('languages with full audio support', ' ').str.replace('*', ' ')
df['is_free'] = df['is_free'].map(lambda x: int(x == True))

df.loc[df['is_free'] == 1, 'genres'] = (
    df['genres']
    .str.replace(r',\s*Free To Play|Free To Play\s*,?', '', regex=True)
)

if 'controller_support' in df.columns:
    df['controller_support'] = df['controller_support'].fillna(0)
    df['controller_support'] = df['controller_support'].astype(str).str.replace('full', '1')

if 'recommendations' in df.columns:
    df['recommendations'] = df['recommendations'].apply(
        lambda x: x.get('total', 0) if isinstance(x, dict) else 0
    )
    
print(df.head(5))

df.to_excel('ex.xlsx')

                            name  is_free  \
0                   Mystic Space        0   
1                      DK Online        1   
2                  MissionX Beta        1   
3                           Evie        0   
4  Winner Winner Chicken Dinner!        0   

                                      about_the_game  \
0  Mystic Space is a fast-paced high action game ...   
1    DK Online - Season 4: Annihilation   Opening...   
2  MissionX - a competitive multiplayer free-roam...   
3         A girl crashed into a dark forest not k...   
4  You were an ordinary young cock like your frie...   

                                 supported_languages    developers  \
0                                            English      EckGames   
1  English   , Korean, Traditional Chinese, Simpl...    Masangsoft   
2    English   , Traditional Chinese, Ukrainian            Holomia   
3                                  English, Japanese  Chilla's Art   
4                                         